Projek AI

Kaggle: https://www.kaggle.com/datasets/aibloy/fairface

Bagian rahma

1. Download Dataset FairFace dari Kaggle

Install & Download Dataset
Cell ini menginstall library `opendatasets` untuk mendownload dataset dari Kaggle, kemudian mengunduh dataset **FairFace** ke lingkungan kerja.

In [1]:
!pip install opendatasets --quiet
import opendatasets as od
import os
od.download("https://www.kaggle.com/datasets/aibloy/fairface")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: amapyuu
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/aibloy/fairface


100%|██████████| 550M/550M [00:31<00:00, 18.4MB/s]


2. Import Library

## 📚 Import Library
Mengimport semua library yang dibutuhkan:
- `os`, `shutil` → operasi file & folder
- `numpy`, `pandas` → manipulasi data
- `matplotlib`, `seaborn` → visualisasi grafik
- `cv2` → membaca dan memproses gambar
- `tqdm` → progress bar
- `sklearn`, `imblearn` → split data dan oversampling

In [2]:
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

from tqdm import tqdm
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler

2. Konfigurasi Kaggle API

Fungsi: Setup Kaggle API
Fungsi `setup_kaggle()` mengkonfigurasi kredensial API Kaggle menggunakan input manual yang aman (tidak hardcode), sehingga program bisa mengakses dataset Kaggle.

In [3]:
def setup_kaggle():

    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)

    import getpass

    username = os.environ.get('KAGGLE_USERNAME') or getpass.getpass("Masukkan Kaggle Username: ")
    key      = os.environ.get('KAGGLE_KEY')      or getpass.getpass("Masukkan Kaggle API Key: ")

    os.environ['KAGGLE_USERNAME'] = username
    os.environ['KAGGLE_KEY']      = key

    print("✅ Kaggle API berhasil dikonfigurasi")


Fungsi: Download Dataset dari Kaggle
Fungsi `download_dataset()` mendownload dataset FairFace dari Kaggle dan menyimpannya ke folder `data/raw`.

In [4]:
def download_dataset(
        dataset_slug="aibloy/fairface",
        output_dir="data/raw"
):

    import kaggle

    os.makedirs(output_dir, exist_ok=True)

    print(f"\n📥 Mendownload dataset: {dataset_slug}")

    kaggle.api.dataset_download_files(
        dataset_slug,
        path=output_dir,
        unzip=True
    )

    print(f"✅ Dataset berhasil didownload ke: {output_dir}")


Fungsi: Load Label Dataset
Fungsi `load_labels()` membaca file CSV label FairFace (train & val), menggabungkannya, dan menambahkan kolom `split` untuk membedakan data train dan validasi.

In [5]:
def load_labels(data_dir="fairface"):

    train_csv = os.path.join(
        data_dir,
        "FairFace",
        "train_labels.csv"
    )

    val_csv = os.path.join(
        data_dir,
        "FairFace",
        "val_labels.csv"
    )

    print("📂 File CSV ditemukan:")
    print(train_csv)
    print(val_csv)

    df_train = pd.read_csv(train_csv)
    df_val   = pd.read_csv(val_csv)

    df_train["split"] = "train"
    df_val["split"]   = "val"

    df = pd.concat(
        [df_train, df_val],
        ignore_index=True
    )

    print(f"\n📊 Data awal: {len(df)} baris")
    print(f"📌 Kolom: {list(df.columns)}")

    print("\n📌 Preview Dataset:")
    print(df.head())

    return df

In [6]:
df = load_labels()

📂 File CSV ditemukan:
fairface/FairFace/train_labels.csv
fairface/FairFace/val_labels.csv

📊 Data awal: 97698 baris
📌 Kolom: ['file', 'age', 'gender', 'race', 'service_test', 'split']

📌 Preview Dataset:
          file    age  gender        race  service_test  split
0  train/1.jpg  50-59    Male  East Asian          True  train
1  train/2.jpg  30-39  Female      Indian         False  train
2  train/3.jpg    3-9  Female       Black         False  train
3  train/4.jpg  20-29  Female      Indian          True  train
4  train/5.jpg  20-29  Female      Indian          True  train


Fungsi: Exploratory Data Analysis (EDA)
Fungsi `run_eda()` menganalisis distribusi data (race, gender, usia) dan mengecek missing values untuk memahami karakteristik dataset sebelum diproses.

In [7]:
def run_eda(df):

    print("\n" + "="*50)
    print("📊 EXPLORATORY DATA ANALYSIS")
    print("="*50)

    print("\n🔍 Missing Values:")
    print(df.isnull().sum())

    # Distribusi Race
    plt.figure(figsize=(10,4))

    sns.countplot(
        data=df,
        x="race",
        order=df["race"].value_counts().index
    )

    plt.title("Distribusi Race")
    plt.xticks(rotation=30)
    plt.show()

    plt.figure(figsize=(5,4))

    sns.countplot(
        data=df,
        x="gender"
    )

    plt.title("Distribusi Gender")
    plt.show()

    plt.figure(figsize=(10,4))

    sns.countplot(
        data=df,
        x="age",
        order=df["age"].value_counts().index
    )

    plt.title("Distribusi Usia")
    plt.xticks(rotation=30)
    plt.show()

    print("\n✅ EDA selesai")

Fungsi: Membersihkan Data CSV
Fungsi `clean_csv()` membersihkan data dari duplikat, missing values, dan spasi berlebih pada kolom kategori.

In [8]:
def clean_csv(df):

    print("\n" + "="*50)
    print("🧹 MEMBERSIHKAN CSV")
    print("="*50)

    awal = len(df)

    df = df.drop_duplicates()

    print(f"   Hapus duplikat       : {awal - len(df)} baris dihapus")

    sebelum = len(df)

    df = df.dropna(
        subset=["file", "race", "gender", "age"]
    )

    print(f"   Hapus missing values : {sebelum - len(df)} baris dihapus")

    for col in ["race", "gender", "age"]:

        df[col] = df[col].astype(str).str.strip()

    df = df.reset_index(drop=True)

    print(f"\n✅ Data bersih: {len(df)} baris")
    print(f"📊 Total data awal: {awal}")

    return df


Mapping & Fungsi: Assign Label Undertone
`UNDERTONE_MAP` memetakan ras ke kategori undertone kulit (cool/neutral/warm). Fungsi `assign_undertone()` menambahkan kolom `undertone` ke dataframe berdasarkan peta ini.

In [9]:
UNDERTONE_MAP = {

    "White": "cool",
    "East Asian": "neutral",
    "Indian": "warm",
    "Black": "warm",
    "Middle Eastern": "warm",
    "Latino_Hispanic": "warm",
    "Southeast Asian": "neutral"

}

def assign_undertone(df):

    df["undertone"] = df["race"].map(
        UNDERTONE_MAP
    ).fillna("neutral")

    print("\n🎨 Distribusi Undertone:")
    print(df["undertone"].value_counts())

    return df

Fungsi: Balancing Data
Fungsi `balance_undertone()` menyeimbangkan jumlah data antar kelas undertone menggunakan **Random Over Sampling** agar model tidak bias ke satu kelas.

In [10]:
def balance_undertone(df):

    X = df.drop(columns=["undertone"])
    y = df["undertone"]

    ros = RandomOverSampler(random_state=42)

    X_resampled, y_resampled = ros.fit_resample(X, y)

    balanced_df = X_resampled.copy()
    balanced_df["undertone"] = y_resampled

    print("\n⚖️ Distribusi Setelah Balancing:")
    print(balanced_df["undertone"].value_counts())

    return balanced_df

Fungsi: Preprocessing Gambar
Fungsi `preprocess_images()` membaca setiap gambar, meresize ke **224×224 piksel**, dan menyimpannya ke folder `data/processed`. Gambar rusak/tidak ditemukan dilewati otomatis.

In [11]:
def preprocess_images(
        df,
        data_dir="data/raw",
        output_dir="data/processed",
        img_size=(224,224)
):

    os.makedirs(output_dir, exist_ok=True)

    valid_rows = []

    print(f"\n🖼️ Memproses {len(df)} gambar...")

    for idx, row in tqdm(
            df.iterrows(),
            total=len(df)
    ):

        img_path = os.path.join(
            data_dir,
            "FairFace",
            str(row["file"])
        )

        img_path = img_path.replace("\\", "/")

        if not os.path.exists(img_path):
            continue

        # Baca gambar
        img = cv2.imread(img_path)

        if img is None:
            continue

        img = cv2.resize(img, img_size)

        save_path = os.path.join(
            output_dir,
            row["file"]
        )

        os.makedirs(
            os.path.dirname(save_path),
            exist_ok=True
        )

        cv2.imwrite(save_path, img)

        valid_rows.append(idx)

    df_final = df.loc[valid_rows].copy()

    df_final["undertone_final"] = df_final["undertone"]

    print("\n✅ Preprocessing selesai")
    print(f"📁 Data disimpan ke: {output_dir}")
    print(f"📊 Total gambar valid: {len(df_final)}")

    return df_final

Eksekusi: Load Data
Memanggil `load_labels()` untuk membaca label dataset ke variabel `df`.

In [12]:
df = load_labels()

df = clean_csv(df)

df = assign_undertone(df)

df = balance_undertone(df)

📂 File CSV ditemukan:
fairface/FairFace/train_labels.csv
fairface/FairFace/val_labels.csv

📊 Data awal: 97698 baris
📌 Kolom: ['file', 'age', 'gender', 'race', 'service_test', 'split']

📌 Preview Dataset:
          file    age  gender        race  service_test  split
0  train/1.jpg  50-59    Male  East Asian          True  train
1  train/2.jpg  30-39  Female      Indian         False  train
2  train/3.jpg    3-9  Female       Black         False  train
3  train/4.jpg  20-29  Female      Indian          True  train
4  train/5.jpg  20-29  Female      Indian          True  train

🧹 MEMBERSIHKAN CSV
   Hapus duplikat       : 0 baris dihapus
   Hapus missing values : 0 baris dihapus

✅ Data bersih: 97698 baris
📊 Total data awal: 97698

🎨 Distribusi Undertone:
undertone
warm       53039
neutral    26047
cool       18612
Name: count, dtype: int64

⚖️ Distribusi Setelah Balancing:
undertone
neutral    53039
warm       53039
cool       53039
Name: count, dtype: int64


In [ ]:
df_final = preprocess_images(df, data_dir="fairface", output_dir="data/processed")


🖼️ Memproses 159117 gambar...


 67%|██████▋   | 106449/159117 [01:42<00:44, 1175.17it/s]

In [ ]:
def save_results(df, output_csv="data/processed/labels_final.csv"):
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    df.to_csv(output_csv, index=False)
    print(f"\n💾 CSV berhasil disimpan")
    print(f"📁 Lokasi: {output_csv}")

In [ ]:
save_results(df_final)

Eksekusi: Preprocessing Gambar & Simpan
Memproses semua gambar (resize & validasi) lalu menyimpan data final ke CSV.

Fungsi: Simpan Hasil ke CSV
Fungsi `save_results()` menyimpan dataframe hasil akhir preprocessing ke file CSV di path yang ditentukan.

In [ ]:
def save_results(
        df,
        output_csv="data/processed/labels_final.csv"
):

    os.makedirs(
        os.path.dirname(output_csv),
        exist_ok=True
    )

    df.to_csv(
        output_csv,
        index=False
    )

    print(f"\n💾 CSV berhasil disimpan")
    print(f"📁 Lokasi: {output_csv}")

Eksekusi: Full Pipeline Rahma
Menjalankan seluruh pipeline dari awal sampai akhir: setup Kaggle, download dataset, load label, EDA, cleaning, assign undertone, balancing, preprocessing gambar, dan simpan hasil CSV.

In [ ]:
os.makedirs("output", exist_ok=True)

setup_kaggle()

download_dataset()

df = load_labels()

print("\n📌 Preview Dataset:")
print(df.head())

run_eda(df)

df = clean_csv(df)

df = assign_undertone(df)

df = df.sample(n=30000, random_state=42)

df = balance_undertone(df)

df_final = preprocess_images(df, data_dir="fairface")

save_results(df_final)

print("\n🎉 Pipeline EDA & Preprocessing selesai!")

Bagian Parida

Import Library (Bagian Model - Parida)
Mengimport library untuk deep learning:
- `tensorflow/keras` → framework utama
- `MobileNetV2` → model pretrained
- `ImageDataGenerator` → augmentasi gambar
- Callbacks → kontrol training
- `sklearn.metrics` → evaluasi model

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    ConfusionMatrixDisplay
)

from sklearn.preprocessing import label_binarize

Konfigurasi Parameter Model
Mendefinisikan hyperparameter dan path:
- `IMG_SIZE` = 224×224, `BATCH_SIZE` = 32, `EPOCHS` = 30
- `NUM_CLASSES` = 3 (warm, cool, neutral)
- Path CSV, folder gambar, dan path simpan model

In [ ]:
IMG_SIZE    = (224, 224)
BATCH_SIZE = 128
EPOCHS     = 5
NUM_CLASSES = 3

LABEL_COL = "undertone_final"

IMG_DIR   = "data/processed"

CSV_PATH  = "data/processed/labels_final.csv"

MODEL_PATH = "model/tinthint_model.h5"

Fungsi: Load Data untuk Training
Fungsi `load_data()` membaca CSV hasil preprocessing, membersihkan label kosong, dan memfilter hanya label valid (warm/cool/neutral).

In [ ]:
def load_data():

    df = pd.read_csv(CSV_PATH)

    df = df.dropna(subset=[LABEL_COL])

    df = df[
        df[LABEL_COL].isin(
            ["warm", "cool", "neutral"]
        )
    ]

    df = df.reset_index(drop=True)

    print(f"\n📊 Total data training: {len(df)}")
    print("\n🎨 Distribusi Label:")
    print(df[LABEL_COL].value_counts())

    return df

## 🔀 Fungsi: Buat Data Generator
Fungsi `build_generators()` membagi dataset (80% train, 10% val, 10% test) dan membuat ImageDataGenerator dengan augmentasi untuk training dan normalisasi saja untuk val/test.

In [ ]:
def build_generators(df):

    train_df, val_df = train_test_split(
        df,
        test_size=0.2,
        stratify=df[LABEL_COL],
        random_state=42
    )

    val_df, test_df = train_test_split(
        val_df,
        test_size=0.5,
        stratify=val_df[LABEL_COL],
        random_state=42
    )

    print("\n📂 Split Dataset")
    print(f"Train : {len(train_df)}")
    print(f"Val   : {len(val_df)}")
    print(f"Test  : {len(test_df)}")

    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        horizontal_flip=True,
        brightness_range=[0.8, 1.2],
        zoom_range=0.1
    )

    val_datagen = ImageDataGenerator(
        rescale=1./255
    )

    def make_generator(datagen,
                       dataframe,
                       shuffle):

        return datagen.flow_from_dataframe(
            dataframe=dataframe,
            directory=IMG_DIR,
            x_col="file",
            y_col=LABEL_COL,
            target_size=IMG_SIZE,
            batch_size=BATCH_SIZE,
            class_mode="categorical",
            shuffle=shuffle
        )

    train_gen = make_generator(
        train_datagen,
        train_df,
        True
    )

    val_gen = make_generator(
        val_datagen,
        val_df,
        False
    )

    test_gen = make_generator(
        val_datagen,
        test_df,
        False
    )

    return train_gen, val_gen, test_gen

Fungsi: Bangun Arsitektur Model
Fungsi `build_model()` membangun CNN berbasis **Transfer Learning MobileNetV2**: layer awal di-freeze, 30 layer terakhir dilatih, ditambah Dense(256) → Dropout → Dense(128) → Dropout → Dense(3, softmax), dikompilasi dengan Adam optimizer.

In [ ]:
def build_model():

    base_model = MobileNetV2(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights="imagenet"
    )

    base_model.trainable = True

    for layer in base_model.layers[:-30]:
        layer.trainable = False

    model = models.Sequential([

        base_model,

        layers.GlobalAveragePooling2D(),

        layers.BatchNormalization(),

        layers.Dense(
            256,
            activation="relu"
        ),

        layers.Dropout(0.4),

        layers.Dense(
            128,
            activation="relu"
        ),

        layers.Dropout(0.3),

        layers.Dense(
            NUM_CLASSES,
            activation="softmax"
        )

    ])

    model.compile(

        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-4
        ),

        loss="categorical_crossentropy",

        metrics=["accuracy"]

    )

    print("\n🧠 Arsitektur Model:")
    model.summary()

    return model

Fungsi: Training Model
Fungsi `train_model()` melatih model dengan 3 callback:
- **EarlyStopping** → berhenti jika tidak ada peningkatan
- **ModelCheckpoint** → simpan model terbaik otomatis
- **ReduceLROnPlateau** → turunkan learning rate jika stagnan

In [ ]:
def train_model(model,
                train_gen,
                val_gen):

    callbacks = [

        EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),

        ModelCheckpoint(
            MODEL_PATH,
            monitor="val_accuracy",
            save_best_only=True,
            verbose=1
        ),

        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        )

    ]

    print(f"\n🚀 Mulai Training ({EPOCHS} Epoch)\n")

    history = model.fit(

        train_gen,

        validation_data=val_gen,

        epochs=EPOCHS,

        callbacks=callbacks,

        verbose=1

    )

    print("\n✅ Training selesai")

    return history

Fungsi: Plot Grafik Training
Fungsi `plot_history()` menampilkan grafik accuracy dan loss (train vs validasi) per epoch untuk mendeteksi overfitting atau underfitting.

In [ ]:
def plot_history(history):

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14,5)
    )

    axes[0].plot(
        history.history["accuracy"],
        label="Train Accuracy"
    )

    axes[0].plot(
        history.history["val_accuracy"],
        label="Validation Accuracy"
    )

    axes[0].set_title(
        "Accuracy Training vs Validation"
    )

    axes[0].legend()

    axes[1].plot(
        history.history["loss"],
        label="Train Loss"
    )

    axes[1].plot(
        history.history["val_loss"],
        label="Validation Loss"
    )

    axes[1].set_title(
        "Loss Training vs Validation"
    )

    axes[1].legend()

    plt.show()

    print("\n✅ Plot training berhasil ditampilkan")

Eksekusi: Main Program (Load → Build)
Menjalankan: load data, buat generator train/val/test, dan bangun arsitektur model. Training dijalankan di cell berikutnya.

In [ ]:
df = load_data()

train_gen, val_gen, test_gen = build_generators(df)

model = build_model()

In [ ]:
print("Sebelum balancing:")
print(df["undertone"].value_counts())

Eksekusi: Training, Evaluasi & Simpan Model
Menjalankan training model, menampilkan grafik hasil training, mengevaluasi performa pada data test, dan menyimpan hasil akhir.

Fungsi: Evaluasi Model
Fungsi `evaluate_model()` mengukur performa model pada data test menggunakan:
- **Classification Report** → precision, recall, F1 per kelas
- **Confusion Matrix** → visualisasi prediksi benar/salah
- **ROC-AUC Score** → kemampuan membedakan antar kelas
- **Test Accuracy** → akurasi keseluruhan

In [ ]:
def evaluate_model(model,
                   test_gen):

    print("\n" + "="*50)
    print("📊 EVALUASI MODEL")
    print("="*50)

    predictions = model.predict(
        test_gen,
        verbose=1
    )

    y_pred = np.argmax(
        predictions,
        axis=1
    )

    y_true = test_gen.classes

    class_names = list(
        test_gen.class_indices.keys()
    )

    print("\n📋 Classification Report:\n")

    print(

        classification_report(
            y_true,
            y_pred,
            target_names=class_names
        )

    )

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )

    fig, ax = plt.subplots(
        figsize=(7,6)
    )

    disp.plot(
        ax=ax,
        colorbar=False
    )

    plt.title("Confusion Matrix")
    plt.show()

    # ROC AUC
    y_true_bin = label_binarize(
        y_true,
        classes=list(range(NUM_CLASSES))
    )

    roc_auc = roc_auc_score(
        y_true_bin,
        predictions,
        multi_class="ovr",
        average="macro"
    )

    print(f"\n🎯 ROC-AUC Score : {roc_auc:.4f}")

    # Accuracy
    accuracy = np.mean(
        y_pred == y_true
    )

    print(f"🎯 Test Accuracy : {accuracy*100:.2f}%")

Fungsi: Prediksi Satu Gambar
Fungsi `predict_image()` memprediksi undertone dari satu gambar baru: load → resize 224×224 → normalisasi → prediksi → tampilkan label undertone dan confidence score (tingkat keyakinan model).

In [ ]:
def predict_image(
        image_path,
        model
):

    from tensorflow.keras.preprocessing import image

    img = image.load_img(
        image_path,
        target_size=IMG_SIZE
    )

    img_array = image.img_to_array(img)

    img_array = img_array / 255.0

    img_array = np.expand_dims(
        img_array,
        axis=0
    )

    prediction = model.predict(img_array)

    predicted_class = np.argmax(
        prediction
    )

    confidence = np.max(
        prediction
    )

    labels = {
        0: "cool",
        1: "neutral",
        2: "warm"
    }

    print("\n🖼️ HASIL PREDIKSI")
    print(f"Undertone : {labels[predicted_class]}")
    print(f"Confidence: {confidence:.4f}")

    return labels[predicted_class]

Contoh Penggunaan: Prediksi Gambar Baru
Cell ini menunjukkan cara memakai fungsi `predict_image()` untuk memprediksi undertone kulit dari satu gambar baru.

In [ ]:
sample_image_path = "data/processed/train/1.jpg"
hasil = predict_image(sample_image_path, model)
print(f"\n✅ Undertone terprediksi: {hasil}")


Download Model yang Sudah Dilatih
Mendownload file model `tinthint_model.h5` dari Google Colab ke komputer lokal untuk disimpan dan dipakai kembali tanpa perlu training ulang.

In [ ]:
from google.colab import files
files.download('model/tinthint_model.h5')